# Verify HU and Plaque Context

Clean Colab workflow for testing whether nnU-Net plaque labels are mostly calcified cores and whether nearby dilated-shell voxels contain lower-attenuation, noncalcified plaque-like tissue.

Research use only. Not clinically validated. Not for diagnosis.

## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install and clone/pull OpenPlaque


In [ ]:
from pathlib import Path
import inspect, os, sys, subprocess
REPO_URL = "https://github.com/pazzani/OpenPlaque.git"
REPO_DIR = Path("/content/OpenPlaque")
REPO_BRANCH = "agent/plaque-context-verification"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "-e", str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=False)
sys.path.insert(0, str(REPO_DIR / "src"))
import openplaque
import openplaque.plaque_context as plaque_context
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
signature = inspect.signature(plaque_context.compute_reports_plaque_context)
assert "vessel_label" in signature.parameters, f"Unexpected plaque_context version from {plaque_context.__file__}: {signature}"
print("Using", REPO_DIR)
print("Branch", branch)
print("Commit", commit)
print("openplaque from", openplaque.__file__)
print("plaque_context signature", signature)


## 3. Configure UCLA study, model, and outputs

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
STUDY_ZIP = DRIVE_ROOT / 'Full_DICOM.zip'
MODEL_ZIP = DRIVE_ROOT / 'models' / 'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
OUTPUT_DIR = DRIVE_ROOT / 'UCLA_Plaque_Context_Verification'
MASK_DIR = OUTPUT_DIR / 'nnunet_masks'
PLOT_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
FALLBACK_SERIES = {'RCA': 1035, 'LCX': 1039, 'LAD': 1043}
VESSELS = ['LAD', 'RCA', 'LCX']
REUSE_SEGMENTATIONS = True
RUN_NNUNET_IF_MISSING = True
RADII_VOXELS = (1, 2, 3)
VESSEL_DILATION_VOXELS = 1
CONTEXT_MIN_HU = 30
CONTEXT_MAX_HU = 350
EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES = True
HEART_SERIES_LIST = [7, 6]
HEART_HU_THRESHOLD = 150
HEART_DOWNSAMPLE = 2
print('Study:', STUDY_ZIP)
print('Outputs:', OUTPUT_DIR)


## 4. Install or expose nnU-Net model

In [ ]:
import zipfile, shutil

NNUNET_RESULTS = Path('/content/nnUNet_results')
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ.setdefault('nnUNet_raw', '/content/nnUNet_raw')
os.environ.setdefault('nnUNet_preprocessed', '/content/nnUNet_preprocessed')

if MODEL_ZIP.exists() and not (NNUNET_RESULTS / 'Dataset001_CCTA_DHM').exists():
    NNUNET_RESULTS.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MODEL_ZIP) as zf:
        zf.extractall(NNUNET_RESULTS)

print('nnUNet_results =', os.environ['nnUNet_results'])

## 5. Load UCLA DICOM study and detect LAD/RCA/LCX

In [ ]:
from openplaque.study import OpenPlaqueStudy
from openplaque.run_new_data import auto_detect_or_fallback_series

study = OpenPlaqueStudy(str(STUDY_ZIP))
series_map = auto_detect_or_fallback_series(study, fallback_series=FALLBACK_SERIES)
series_map = {k: int(series_map[k]) for k in VESSELS}
series_map

## 6. Run or reuse nnU-Net segmentations

In [ ]:
import numpy as np
import SimpleITK as sitk
from openplaque.segmentation import SegmentationReport, export_for_nnunet, run_nnunet, load_segmentation

reports = []
for vessel in VESSELS:
    image, volume, _ = study.load_series(series_map[vessel])
    saved_mask = MASK_DIR / f'{vessel}.nii.gz'
    if REUSE_SEGMENTATIONS and saved_mask.exists():
        mask_image = sitk.ReadImage(str(saved_mask))
        mask = sitk.GetArrayFromImage(mask_image)
        input_dir = str(OUTPUT_DIR / f'{vessel}_input')
        output_dir = str(MASK_DIR)
        print(f'Reused {saved_mask}')
    elif RUN_NNUNET_IF_MISSING:
        input_dir = str(OUTPUT_DIR / f'{vessel}_input')
        output_dir = str(OUTPUT_DIR / f'{vessel}_nnunet_output')
        export_for_nnunet(image, input_dir, vessel)
        run_nnunet(input_dir, output_dir)
        mask_image, mask = load_segmentation(output_dir, vessel)
        sitk.WriteImage(mask_image, str(saved_mask))
        print(f'Ran nnU-Net and saved {saved_mask}')
    else:
        raise FileNotFoundError(f'Missing saved mask for {vessel}: {saved_mask}')

    reports.append(SegmentationReport(vessel, image, mask_image, volume, mask, input_dir, output_dir))

[(r.name, r.label_counts(), r.hu_statistics()) for r in reports]

## 7. Compute plaque-core and shell HU summaries

In [ ]:
import importlib
import pandas as pd
import openplaque.plaque_context as plaque_context
plaque_context = importlib.reload(plaque_context)
compute_reports_plaque_context = plaque_context.compute_reports_plaque_context
save_context_csv = plaque_context.save_context_csv
write_context_html_report = plaque_context.write_context_html_report

geometric_rows = compute_reports_plaque_context(
    reports,
    radii_voxels=RADII_VOXELS,
    label=2,
    connectivity=26,
)
for row in geometric_rows:
    row['context_mode'] = 'geometric_shell'

filtered_rows = compute_reports_plaque_context(
    reports,
    radii_voxels=RADII_VOXELS,
    label=2,
    vessel_label=1,
    connectivity=26,
    anatomical_filter=True,
    vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
    candidate_min_hu=CONTEXT_MIN_HU,
    candidate_max_hu=CONTEXT_MAX_HU,
    include_candidate_rows=True,
    include_vessel_candidate_rows=True,
    exclude_vessel_label_for_candidates=EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES,
)
for row in filtered_rows:
    row['context_mode'] = 'anatomy_filtered'

rows = geometric_rows + filtered_rows
df = pd.DataFrame(rows)
csv_path = save_context_csv(rows, OUTPUT_DIR / 'plaque_hu_context_summary.csv')
summary_cols = ['context_mode', 'vessel', 'region', 'voxel_count', 'mean_hu', 'hu_30_130_fraction', 'hu_130_350_fraction', 'hu_350_700_fraction', 'hu_700_1000_fraction', 'gt_1000_fraction']
df.sort_values(['context_mode', 'vessel', 'radius_voxels', 'region'])[summary_cols]


## 8. Generate histograms and overlays

In [ ]:
import importlib
import openplaque.plaque_context as plaque_context
plaque_context = importlib.reload(plaque_context)
save_hu_histogram_png = plaque_context.save_hu_histogram_png
save_context_overlay_png = plaque_context.save_context_overlay_png
save_vessel_candidate_overlay_png = plaque_context.save_vessel_candidate_overlay_png
save_candidate_mask_nifti = plaque_context.save_candidate_mask_nifti
vessel_context_mask = plaque_context.vessel_context_mask
vessel_wide_candidate_mask = plaque_context.vessel_wide_candidate_mask

histogram_paths = []
overlay_paths = []
candidate_mask_paths = []
for report in reports:
    include_mask = vessel_context_mask(
        report.mask,
        vessel_label=1,
        plaque_label=2,
        vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
        connectivity=26,
    )
    vessel_candidate = vessel_wide_candidate_mask(
        report.volume,
        report.mask,
        vessel_label=1,
        plaque_label=2,
        vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
        connectivity=26,
        min_hu=CONTEXT_MIN_HU,
        max_hu=CONTEXT_MAX_HU,
        exclude_vessel_label=EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES,
    )
    candidate_mask_paths.append(save_candidate_mask_nifti(
        vessel_candidate,
        report.mask_image,
        MASK_DIR / f'{report.name}_vessel_wide_candidate_30_350HU.nii.gz',
    ))
    histogram_paths.append(save_hu_histogram_png(
        report.volume,
        report.mask,
        PLOT_DIR / f'{report.name}_anatomy_filtered_context_histogram.png',
        f'{report.name} anatomy-filtered plaque context HU',
        radii_voxels=RADII_VOXELS,
        include_mask=include_mask,
    ))
    overlay_paths.append(save_context_overlay_png(
        report.volume,
        report.mask,
        PLOT_DIR / f'{report.name}_context_candidate_overlay.png',
        f'{report.name} calcium-adjacent candidate: {CONTEXT_MIN_HU}-{CONTEXT_MAX_HU} HU',
        radius_voxels=3,
        include_mask=include_mask,
        candidate_min_hu=CONTEXT_MIN_HU,
        candidate_max_hu=CONTEXT_MAX_HU,
    ))
    overlay_paths.append(save_vessel_candidate_overlay_png(
        report.volume,
        report.mask,
        PLOT_DIR / f'{report.name}_vessel_wide_candidate_overlay.png',
        f'{report.name} vessel-wide candidate: {CONTEXT_MIN_HU}-{CONTEXT_MAX_HU} HU',
        vessel_label=1,
        plaque_label=2,
        vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
        min_hu=CONTEXT_MIN_HU,
        max_hu=CONTEXT_MAX_HU,
        exclude_vessel_label=EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES,
    ))

html_path = write_context_html_report(
    OUTPUT_DIR / 'plaque_hu_context_report.html',
    rows,
    histogram_paths=histogram_paths,
    overlay_paths=overlay_paths,
)
print('CSV:', csv_path)
print('HTML:', html_path)
print('Plots:', PLOT_DIR)
print('Candidate masks:', candidate_mask_paths)


## 9. Interactive 3D candidate visualization

Rotatable Plotly views: red = nnU-Net plaque core, green = vessel-wide `30-350 HU` candidate noncalcified/mixed plaque. HTML files are saved to Drive for later review.


In [ ]:
import importlib
import numpy as np
import plotly.graph_objects as go
from skimage import measure
import openplaque.plaque_context as plaque_context
plaque_context = importlib.reload(plaque_context)
vessel_wide_candidate_mask = plaque_context.vessel_wide_candidate_mask

def _mesh_trace(mask, name, color, spacing_xyz=(1.0, 1.0, 1.0), opacity=0.55):
    mask = np.asarray(mask, dtype=bool)
    if int(mask.sum()) == 0:
        return None
    padded = np.pad(mask.astype(np.uint8), 1, mode="constant")
    spacing_zyx = (float(spacing_xyz[2]), float(spacing_xyz[1]), float(spacing_xyz[0]))
    verts, faces, _, _ = measure.marching_cubes(padded, level=0.5, spacing=spacing_zyx)
    # Undo one-voxel padding in physical coordinates. marching_cubes returns z, y, x.
    verts[:, 0] -= spacing_zyx[0]
    verts[:, 1] -= spacing_zyx[1]
    verts[:, 2] -= spacing_zyx[2]
    return go.Mesh3d(
        x=verts[:, 2], y=verts[:, 1], z=verts[:, 0],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        name=name, color=color, opacity=opacity, flatshading=True,
        hoverinfo="name", showscale=False,
    )

def make_3d_plaque_figure(report):
    plaque = report.mask == 2
    candidate = vessel_wide_candidate_mask(
        report.volume, report.mask,
        vessel_label=1, plaque_label=2,
        vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
        connectivity=26,
        min_hu=CONTEXT_MIN_HU, max_hu=CONTEXT_MAX_HU,
        exclude_vessel_label=EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES,
    )
    spacing_xyz = report.mask_image.GetSpacing()
    traces = [
        _mesh_trace(candidate, f"{report.name} vessel-wide 30-350 HU candidate", "limegreen", spacing_xyz, opacity=0.45),
        _mesh_trace(plaque, f"{report.name} nnU-Net plaque core", "red", spacing_xyz, opacity=0.75),
    ]
    traces = [t for t in traces if t is not None]
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f"{report.name}: rotatable plaque core and vessel-wide candidate",
        scene=dict(
            xaxis_title="x mm", yaxis_title="y mm", zaxis_title="z mm",
            aspectmode="data",
        ),
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig

three_d_paths = []
figures_3d = {}
for report in reports:
    fig = make_3d_plaque_figure(report)
    figures_3d[report.name] = fig
    path = PLOT_DIR / f"{report.name}_interactive_3d_plaque_context.html"
    fig.write_html(str(path), include_plotlyjs="cdn")
    three_d_paths.append(path)

print("3D HTML files:")
for path in three_d_paths:
    print(path)

# Show LAD inline by default; change the key to RCA or LCX if desired.
figures_3d.get("LAD", next(iter(figures_3d.values()))).show()


## 10. Interactive 3D heart and plaque phase comparison

Loads DICOM series 7 and 6 as heart/CTA reference volumes and renders translucent high-HU anatomy surfaces with plaque/candidate surfaces in patient physical coordinates. Compare the saved HTML files to see which cardiac phase aligns better with the curved-vessel plaque outputs. If the reformats are not spatially registered to the axial CTA series, plaque surfaces may not align anatomically.


In [ ]:
import importlib
import numpy as np
import plotly.graph_objects as go
from skimage import measure
import openplaque.plaque_context as plaque_context
plaque_context = importlib.reload(plaque_context)
vessel_wide_candidate_mask = plaque_context.vessel_wide_candidate_mask

def _mesh_trace_physical(mask, image, name, color, opacity=0.4, step=1):
    mask = np.asarray(mask, dtype=bool)
    if step > 1:
        mask = mask[::step, ::step, ::step]
    if int(mask.sum()) == 0:
        return None
    padded = np.pad(mask.astype(np.uint8), 1, mode="constant")
    spacing_xyz = np.asarray(image.GetSpacing(), dtype=float) * float(step)
    spacing_zyx = (spacing_xyz[2], spacing_xyz[1], spacing_xyz[0])
    verts_zyx, faces, _, _ = measure.marching_cubes(padded, level=0.5, spacing=spacing_zyx)
    verts_zyx[:, 0] -= spacing_zyx[0]
    verts_zyx[:, 1] -= spacing_zyx[1]
    verts_zyx[:, 2] -= spacing_zyx[2]
    xyz_scaled = np.column_stack([verts_zyx[:, 2], verts_zyx[:, 1], verts_zyx[:, 0]])
    direction = np.asarray(image.GetDirection(), dtype=float).reshape(3, 3)
    origin = np.asarray(image.GetOrigin(), dtype=float)
    phys = origin + xyz_scaled @ direction.T
    return go.Mesh3d(
        x=phys[:, 0], y=phys[:, 1], z=phys[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        name=name, color=color, opacity=opacity, flatshading=True,
        hoverinfo="name", showscale=False,
    )

def make_heart_plaque_figure(heart_series):
    heart_image, heart_volume, _ = study.load_series(int(heart_series))
    heart_mask = heart_volume >= HEART_HU_THRESHOLD
    traces = [
        _mesh_trace_physical(
            heart_mask, heart_image,
            f"Series {heart_series} high-HU anatomy >= {HEART_HU_THRESHOLD} HU",
            "lightgray", opacity=0.16, step=HEART_DOWNSAMPLE,
        )
    ]
    colors = {"LAD": "red", "RCA": "orange", "LCX": "magenta"}
    candidate_colors = {"LAD": "limegreen", "RCA": "cyan", "LCX": "yellowgreen"}
    for report in reports:
        plaque = report.mask == 2
        candidate = vessel_wide_candidate_mask(
            report.volume, report.mask,
            vessel_label=1, plaque_label=2,
            vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
            connectivity=26,
            min_hu=CONTEXT_MIN_HU, max_hu=CONTEXT_MAX_HU,
            exclude_vessel_label=EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES,
        )
        traces.append(_mesh_trace_physical(candidate, report.mask_image, f"{report.name} vessel candidate", candidate_colors.get(report.name, "lime"), opacity=0.45))
        traces.append(_mesh_trace_physical(plaque, report.mask_image, f"{report.name} plaque core", colors.get(report.name, "red"), opacity=0.85))
    traces = [t for t in traces if t is not None]
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f"Series {heart_series} heart/CTA with plaque context surfaces",
        scene=dict(
            xaxis_title="patient x mm",
            yaxis_title="patient y mm",
            zaxis_title="patient z mm",
            aspectmode="data",
        ),
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig

heart_3d_paths = []
heart_figures = {}
for heart_series in HEART_SERIES_LIST:
    fig = make_heart_plaque_figure(heart_series)
    heart_figures[int(heart_series)] = fig
    path = PLOT_DIR / f"series_{int(heart_series)}_heart_with_plaque_context_3d.html"
    fig.write_html(str(path), include_plotlyjs="cdn")
    heart_3d_paths.append(path)

print("Heart + plaque 3D HTML files:")
for path in heart_3d_paths:
    print(path)

# Show the first configured heart series inline; open saved HTML files to compare phases.
heart_figures[int(HEART_SERIES_LIST[0])].show()


## 11. Quick interpretation table

`context_candidate_*vox` rows are constrained to tissue adjacent to calcified plaque cores. `vessel_candidate` rows search the broader local vessel anatomy for `30-350 HU` candidate noncalcified/mixed plaque even when it is not touching calcium.

In [ ]:
display_cols = [
    'context_mode', 'vessel', 'region', 'voxel_count', 'mean_hu', 'median_hu',
    'lt_30_fraction', 'hu_30_130_fraction', 'hu_130_350_fraction',
    'hu_350_700_fraction', 'hu_700_1000_fraction', 'gt_1000_fraction',
]
df.sort_values(['context_mode', 'vessel', 'radius_voxels', 'region'])[display_cols]
